# 01 · Data preparation (2016–2021)
Builds `data/Helsinki_MultiYear.csv` and `data/Helsinki_dataQC.csv` from the one-minute FMI measurements of the Helsinki-Kumpula PV system.

**Raw data.** Download the FMI dataset (Karhu & Lindfors, 2025, https://doi.org/10.57707/fmi-b2share.c7b3ec9f19324a6497639d6d8c0cde84) and place the Helsinki file at `data/raw/FMI_Helsinki_PV.csv`. Both output files are already included in the repository, so this notebook only needs to be run to rebuild them.

**Processing.** Night values of power and irradiance are set to zero; daytime drop-outs are treated as gaps, not zeros. Gaps up to three hours are interpolated linearly; longer gaps are filled with the mean of the same hour within ±7 days. Every filled value is flagged in a `<column>_Imputed` column.

## Setup

In [ ]:
import os, sys

REPO = "helsinki-multihorizon-pv"
REPO_URL = f"https://github.com/rhmano/{REPO}.git"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO):
        os.system(f"git clone -q {REPO_URL}")
    ROOT = os.path.abspath(REPO)
    os.system(f"pip install -q pvlib")
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

import logging, warnings
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from pvlib.solarposition import get_solarposition
from pvlib.location import Location

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
log = logging.getLogger("helsinki")


def section(title: str, char: str = "=", width: int = 72) -> None:
    log.info(char * width)
    log.info(title)
    log.info(char * width)

## Configuration

In [ ]:
@dataclass(frozen=True)
class HelsinkiConfig:
    raw_path: str = os.path.join(ROOT, "data", "raw", "FMI_Helsinki_PV.csv")
    output_path: str = os.path.join(ROOT, "data", "Helsinki_MultiYear.csv")
    dataqc_path: str = os.path.join(ROOT, "data", "Helsinki_dataQC.csv")
    header_row: int = 67
    separator: str = ";"
    years: tuple = (2016, 2017, 2018, 2019, 2020, 2021)
    site_name: str = "Helsinki"
    nominal_capacity_w: float = 20_000.0
    latitude: float = 60.204
    longitude: float = 24.961
    tilt_deg: float = 15.0
    source_columns: dict = field(default_factory=lambda: {
        "pv_inv_out": "Power_Actual_W",
        "GLOBA_PT1M_AVG(:31)": "Irradiance_POA",
        "GLOB_PT1M_AVG": "GHI",
        "TA_PT1M_AVG(:31)": "Temp_Ambient",
        "WS_PT10M_AVG": "Wind_Speed",
        "RH_PT1M_AVG": "Humidity",
        "TTECH_PT1M_AVG(:32)": "Temp_Module",
        "vis_SnoP": "Snow_Indicator",
    })
    model_features: tuple = ("Irradiance_POA", "GHI", "Temp_Ambient", "Wind_Speed", "Humidity")
    diagnostic_only: tuple = ("Temp_Module", "Solar_Elevation", "Clearness_Index", "Snow_Indicator")
    max_irradiance: float = 1400.0
    temp_bounds: tuple = (-45.0, 40.0)
    wind_bounds: tuple = (0.0, 40.0)
    dropout_min_elevation: float = 8.0
    interpolate_limit_hours: int = 3
    climatology_window_days: int = 7


CFG = HelsinkiConfig()
log.info("Years: %s | inverter capacity: %s W", CFG.years, f"{CFG.nominal_capacity_w:,.0f}")

## 1. Loading and hourly aggregation

In [ ]:
def load_raw(cfg: HelsinkiConfig) -> pd.DataFrame:
    section("📥 Loading the raw FMI file")
    if not os.path.exists(cfg.raw_path):
        raise FileNotFoundError(f"{cfg.raw_path} not found — see the download note at the top")
    df = pd.read_csv(cfg.raw_path, sep=cfg.separator, header=cfg.header_row,
                     parse_dates=["utctime"], index_col="utctime",
                     dtype={"Remarks": str}, low_memory=False)
    log.info("Raw size: %s rows × %d columns", f"{len(df):,}", df.shape[1])
    log.info("Range: %s → %s", df.index.min(), df.index.max())
    missing = [c for c in cfg.source_columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing source columns: {missing}")
    out = df[list(cfg.source_columns)].rename(columns=cfg.source_columns)
    out = out[~out.index.duplicated(keep="first")].sort_index()
    hourly = out.resample("1h").mean()
    log.info("After hourly aggregation: %s hours", f"{len(hourly):,}")
    by_year = hourly.groupby(hourly.index.year).apply(lambda g: g.isnull().mean() * 100)
    log.info("Missing share per year (%%):\n%s", by_year.round(1).to_string())
    return hourly


df_hourly = load_raw(CFG)

## 2. Physical cleaning and outage flags
Night is a true zero; a daytime drop-out is a gap. Solar elevation (pvlib) separates the two, which matters at 60° N where the day lasts from about 6 to 19 hours.

In [ ]:
def clean_and_flag(df: pd.DataFrame, cfg: HelsinkiConfig) -> pd.DataFrame:
    section("🧹 Physical cleaning")
    df = df.copy()
    idx = df.index.tz_localize("UTC") if df.index.tz is None else df.index
    df["_elevation"] = get_solarposition(idx, cfg.latitude, cfg.longitude)["elevation"].values
    df["Solar_Elevation"] = df["_elevation"]
    night = df["_elevation"] < 0
    log.info("Daytime hours: %s | night: %s (%.1f%%)",
             f"{int((~night).sum()):,}", f"{int(night.sum()):,}", night.mean() * 100)
    for col in ("Irradiance_POA", "GHI", "Power_Actual_W"):
        n_neg = int((df[col] < 0).sum())
        df[col] = df[col].clip(lower=0)
        if col != "Power_Actual_W":
            df.loc[df[col] > cfg.max_irradiance, col] = np.nan
        log.info("   %-16s %s negative values clipped", col, f"{n_neg:,}")
    lo, hi = cfg.temp_bounds
    for c in ("Temp_Ambient", "Temp_Module"):
        df[c] = df[c].where(df[c].between(lo, hi))
    lo, hi = cfg.wind_bounds
    df["Wind_Speed"] = df["Wind_Speed"].where(df["Wind_Speed"].between(lo, hi))
    df["Humidity"] = df["Humidity"].where(df["Humidity"].between(0, 100))
    for col in ("Irradiance_POA", "GHI", "Power_Actual_W"):
        df.loc[night, col] = 0.0
    section("🔎 Daytime drop-outs")
    day_up = df["_elevation"] > cfg.dropout_min_elevation
    for col in ("Irradiance_POA", "GHI"):
        dropout = day_up & (df[col] < 1)
        df.loc[dropout, col] = np.nan
        if dropout.any():
            by_year = dropout.groupby(df.index.year).sum()
            log.info("   %-16s %s hours | by year: %s", col, f"{int(dropout.sum()):,}",
                     {int(y): int(n) for y, n in by_year.items() if n})
    cs = Location(cfg.latitude, cfg.longitude, tz="UTC").get_clearsky(idx, model="ineichen")
    cs_ghi = cs["ghi"].values
    df["Clearness_Index"] = np.where(
        cs_ghi > 20,
        np.clip(df["GHI"].to_numpy(dtype=float) / np.maximum(cs_ghi, 1.0), 0, 1.2),
        0.0)
    log.info("Clearness index: daytime median %.3f | range %.2f → %.2f",
             float(np.nanmedian(df.loc[cs_ghi > 20, "Clearness_Index"])),
             float(df["Clearness_Index"].min()), float(df["Clearness_Index"].max()))
    log.info("Solar elevation: %.1f° → %.1f°",
             float(df["Solar_Elevation"].min()), float(df["Solar_Elevation"].max()))
    log.info("Distinct snow-indicator values: %s", np.sort(df["Snow_Indicator"].dropna().unique())[:12])
    cols = list(cfg.model_features) + ["Power_Actual_W"]
    log.info("After cleaning:\n%s", df[cols].describe().T[["min", "mean", "max"]].round(2).to_string())
    return df


df_clean = clean_and_flag(df_hourly, CFG)

## 3. Building the six years and filling gaps

In [ ]:
def impute_with_tracking(df: pd.DataFrame, cols, cfg: HelsinkiConfig) -> pd.DataFrame:
    df = df.copy()
    hours, doys = df.index.hour, df.index.dayofyear
    for col in cols:
        flag = df[col].isnull()
        df[f"{col}_Imputed"] = flag
        if not flag.any():
            continue
        df[col] = df[col].interpolate(method="linear", limit=cfg.interpolate_limit_hours)
        for idx in df.index[df[col].isnull()]:
            near = (hours == idx.hour) & (np.abs(doys - idx.dayofyear) <= cfg.climatology_window_days)
            df.loc[idx, col] = df.loc[near, col].mean()
        log.info("      %-16s %s rows filled (%.2f%%)", col, f"{int(flag.sum()):,}", flag.mean() * 100)
    return df


def build_years(df: pd.DataFrame, cfg: HelsinkiConfig) -> pd.DataFrame:
    section(f"📅 Building {len(cfg.years)} years")
    all_cols = list(cfg.model_features) + ["Power_Actual_W"] + list(cfg.diagnostic_only)
    frames = []
    for year in cfg.years:
        full = pd.date_range(f"{year}-01-01 00:00", f"{year}-12-31 23:00", freq="h", tz=df.index.tz)
        y = df[df.index.year == year].reindex(full)
        log.info("◆ %d — available hours %s / %s", year,
                 f"{int(y['Power_Actual_W'].notna().sum()):,}", f"{len(full):,}")
        y = impute_with_tracking(y, all_cols, cfg)
        y["Site_ID"] = cfg.site_name
        y["Year"] = year
        frames.append(y)
    out = pd.concat(frames).sort_index()
    out["Power_PU"] = (out["Power_Actual_W"] / cfg.nominal_capacity_w).clip(0, 1)
    hour, month = out.index.hour, out.index.month
    out["HOUR_SIN"] = np.sin(2 * np.pi * hour / 24.0)
    out["HOUR_COS"] = np.cos(2 * np.pi * hour / 24.0)
    out["MONTH_SIN"] = np.sin(2 * np.pi * month / 12.0)
    out["MONTH_COS"] = np.cos(2 * np.pi * month / 12.0)
    out = out.drop(columns=["_elevation"], errors="ignore")
    out.index.name = "utctime"
    out = out.reset_index()
    schema = (["utctime", "Site_ID", "Year", "Power_Actual_W", "Power_PU"]
              + list(cfg.model_features)
              + ["HOUR_SIN", "HOUR_COS", "MONTH_SIN", "MONTH_COS"]
              + list(cfg.diagnostic_only)
              + [c for c in out.columns if c.endswith("_Imputed")])
    out = out[[c for c in schema if c in out.columns]]
    log.info("Final size: %s rows × %d columns", f"{len(out):,}", out.shape[1])
    return out


df_final = build_years(df_clean, CFG)
df_final.head(3)

## 4. Year-to-year stability of the system

In [ ]:
section("🔬 Year-to-year stability")
day = df_final[df_final["Irradiance_POA"] > 50].copy()
day["eff"] = day["Power_Actual_W"] / day["Irradiance_POA"]
stats = pd.DataFrame({
    "Performance_ratio_median": day.groupby("Year")["eff"].median(),
    "Peak_power_W": df_final.groupby("Year")["Power_Actual_W"].max(),
    "Peak_factor_pct": df_final.groupby("Year")["Power_Actual_W"].max() / CFG.nominal_capacity_w * 100,
    "Corr_POA_power": day.groupby("Year").apply(lambda g: g["Irradiance_POA"].corr(g["Power_Actual_W"])),
    "Annual_POA_kWh_m2": df_final.groupby("Year")["Irradiance_POA"].sum() / 1000,
    "Imputed_POA_pct": df_final.groupby("Year")["Irradiance_POA_Imputed"].mean() * 100,
}).round(3)
print(stats.to_string())
cv = stats["Performance_ratio_median"].std() / stats["Performance_ratio_median"].median()
print(f"\nCoefficient of variation of the performance ratio across years: {cv:.3f}")

## 5. Final audit

In [ ]:
section("🔬 Final audit")
core = ["Power_Actual_W", "Power_PU"] + list(CFG.model_features)
print(f"Missing values in core columns: {int(df_final[core].isnull().sum().sum()):,}")
print("\nRows per year:")
print(df_final["Year"].value_counts().sort_index().to_string())
print("\nImputed share per column:")
for c in [c for c in df_final.columns if c.endswith("_Imputed")]:
    print(f"   {c.replace('_Imputed', ''):20s} {df_final[c].mean() * 100:6.2f}%")
odd_a = len(df_final[(df_final["Irradiance_POA"] < 5) & (df_final["Power_Actual_W"] > 50)])
odd_b = len(df_final[(df_final["Irradiance_POA"] > 300) & (df_final["Power_Actual_W"] < 50)])
print(f"\nPower with near-zero irradiance: {odd_a:,} rows")
print(f"High irradiance without power: {odd_b:,} rows (expected in winter: snow on the modules)")

## 6. Saving the hourly data set

In [ ]:
os.makedirs(os.path.dirname(CFG.output_path), exist_ok=True)
df_final.to_csv(CFG.output_path, index=False)
log.info("Saved %s (%s rows × %d columns)", CFG.output_path, f"{len(df_final):,}", df_final.shape[1])

## 7. Daily FMI quality flag (`dataQC`)
FMI marks days with missing data or serious problems with `dataQC = 0`. The experiment excludes the measured hours of these days.

In [ ]:
raw = pd.read_csv(CFG.raw_path, sep=CFG.separator, header=CFG.header_row,
                  usecols=["utctime", "dataQC"], low_memory=False)
raw["utctime"] = pd.to_datetime(raw["utctime"], utc=True).dt.tz_localize(None)
raw["day"] = raw["utctime"].dt.normalize()
qc = raw.groupby("day")["dataQC"].min().loc["2016-01-01":"2021-12-31"].reset_index()
qc.to_csv(CFG.dataqc_path, index=False)
print("dataQC values:", qc["dataQC"].value_counts(dropna=False).to_dict())
print("Saved", CFG.dataqc_path)

## 8. FMI flag versus the outage rule of the experiment (diagnostic)

In [ ]:
from multistep_core import CFG as MCFG, prepare_base

BASE = prepare_base(MCFG)
day_idx = BASE.index.normalize()
rule = pd.DataFrame({
    "outage_hours": BASE["FLAG_OUTAGE"].groupby(day_idx).sum(),
    "imputed_daytime_hours": (BASE["FLAG_IMPUTED"]
                              & (BASE["GHI_CS"] > MCFG.daytime_cs_threshold)).groupby(day_idx).sum(),
})
K = qc.set_index("day").join(rule, how="left")
K["dataQC_0"] = K["dataQC"] == 0
K["rule_outage"] = K["outage_hours"] > 0
print("Rows: dataQC = 0 | Columns: outage rule")
print(pd.crosstab(K["dataQC_0"], K["rule_outage"], margins=True))
K[K["dataQC_0"] | K["rule_outage"]].to_csv(os.path.join(MCFG.results_dir, "dataqc_comparison.csv"))